# Qwen3-4B — Entraînement QLoRA sur Mac

Dans VS Code, sélectionnez le noyau **TuniScope (notebooks)**, puis cliquez sur **Tout exécuter**. Le notebook prépare les 4 000 exemples du train, entraîne un adaptateur partagé par TSAC et MADAR pendant deux époques, puis évalue le modèle en few-shot sur la validation v2.

Prévoyez environ **2 h 15 à 2 h 30**. La sortie de la cellule d’entraînement affiche le pourcentage, la perte et le temps restant estimé. En cas d’interruption, relancez **Tout exécuter** avec le même dossier pour reprendre au dernier point sauvegardé.

In [1]:
from collections import deque
import json
import os
from pathlib import Path
import subprocess

from IPython.display import Markdown, display

HERE = Path.cwd().resolve()
ROOT = next((path for path in (HERE, *HERE.parents)
             if (path / 'scripts/run_qwen4b_qlora.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Ouvrez ce notebook depuis le dossier du projet dans VS Code.')

RUN = ROOT / 'outputs/qwen/qwen3_4b_qlora_full_v1'
MLX_PYTHON = ROOT / 'environments/qwen4b/.venv/bin/python'
SCRIPT = ROOT / 'scripts/run_qwen4b_qlora.py'
PLAN = json.loads((ROOT / 'configs/qwen3_4b_transition_v1.json').read_text())
BASE = ROOT / '.tools/llms' / f"qwen3_4b_{PLAN['model']['revision']}_mlx_4bit"
if not MLX_PYTHON.is_file() or not (BASE / 'conversion.json').is_file():
    raise RuntimeError('Préparez le profil MLX et la base 4 bits comme indiqué dans docs/81_transition_qwen3_4b.md.')

ENV = dict(os.environ, HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', TOKENIZERS_PARALLELISM='false')
ENV.pop('VIRTUAL_ENV', None)
RUN.mkdir(parents=True, exist_ok=True)

def lancer(action, titre):
    if action == 'summarize':
        python = ['sh', str(ROOT / 'scripts/project_uv.sh'), 'run', '--locked', '--offline', 'python']
    else:
        python = [str(MLX_PYTHON), '-u']
    commande = [*python, str(SCRIPT), action, '--output', str(RUN)]
    journal = RUN / f'{action}.log'
    lignes = deque(maxlen=8)
    vue = display(Markdown(f'**{titre}**'), display_id=True)
    with journal.open('a', encoding='utf-8') as fichier, subprocess.Popen(
        commande, cwd=ROOT, env=ENV, text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, start_new_session=True,
    ) as processus:
        try:
            for ligne in processus.stdout:
                fichier.write(ligne)
                fichier.flush()
                lignes.append(ligne.rstrip())
                recent = '\n'.join('    ' + texte for texte in lignes)
                vue.update(Markdown(f'**{titre}**\n\n{recent}'))
            code = processus.wait()
        except KeyboardInterrupt:
            if processus.poll() is None:
                processus.terminate()
            print('Interrompu. Relancez cette cellule pour reprendre depuis la dernière sauvegarde.')
            raise
    if code:
        raise RuntimeError(f'Échec de la commande. Consultez le journal {journal}.')

print(f'Dossier du run : {RUN}')

Dossier du run : /Users/elghoth/Desktop/NLP Project/outputs/qwen/qwen3_4b_qlora_full_v1


## 1. Entraîner l’adaptateur

La perte mesure l’écart entre les réponses du modèle et les étiquettes attendues sur le train. La cellule sauvegarde régulièrement la progression, puis écrit l’adaptateur final.

In [ ]:
lancer('train', 'Entraînement QLoRA')

**Entraînement QLoRA**

    [###################-]  95.2% | époque 2/2 | 7616/8000 | loss 0.0924 | écoulé 01:45:00 | reste estimé 00:05:18
    [###################-]  95.6% | époque 2/2 | 7648/8000 | loss 0.0717 | écoulé 01:45:26 | reste estimé 00:04:51
    [###################-]  96.0% | époque 2/2 | 7680/8000 | loss 0.0953 | écoulé 01:45:53 | reste estimé 00:04:25
    [###################-]  96.4% | époque 2/2 | 7712/8000 | loss 0.0581 | écoulé 01:46:19 | reste estimé 00:03:58
    [###################-]  96.8% | époque 2/2 | 7744/8000 | loss 0.0995 | écoulé 01:46:46 | reste estimé 00:03:32
    [###################-]  97.2% | époque 2/2 | 7776/8000 | loss 0.1269 | écoulé 01:47:12 | reste estimé 00:03:05
    [###################-]  97.6% | époque 2/2 | 7808/8000 | loss 0.0447 | écoulé 01:47:41 | reste estimé 00:02:39
    [###################-]  98.0% | époque 2/2 | 7840/8000 | loss 0.0987 | écoulé 01:48:07 | reste estimé 00:02:12

## 2. Évaluer et comparer

Le modèle adapté reçoit les mêmes exemples few-shot et les mêmes textes de validation que le modèle de base. La dernière cellule affiche l’exactitude et le macro-F1 avant et après adaptation.

In [ ]:
lancer('evaluate', 'Validation few-shot TSAC et MADAR')
lancer('summarize', 'Calcul des métriques')
resultats = json.loads((RUN / 'comparison.json').read_text())['scores']
tableau = [
    '| Corpus | Textes | Exactitude avant → après | Macro-F1 avant → après | Écart de macro-F1 |',
    '| --- | ---: | ---: | ---: | ---: |',
]
for corpus, resultat in resultats.items():
    base, adapte = resultat['base'], resultat['qlora']
    tableau.append(
        f"| {corpus.upper()} | {resultat['n']} | {base['accuracy']:.2%} → {adapte['accuracy']:.2%} | "
        f"{base['macro_f1']:.2%} → {adapte['macro_f1']:.2%} | {resultat['macro_f1_gain_points']:+.2f} points |"
    )
display(Markdown('\n'.join(tableau)))
print(f'Adaptateur, métriques et journaux : {RUN}')